In [ ]:
!pip install ucimlrepo -q
!pip install scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay)


In [ ]:
ionosphere = fetch_ucirepo(id=52)
X_raw = ionosphere.data.features
y_raw = ionosphere.data.targets

print("Shape dos atributos:", X_raw.shape)
print("\nDistribuição das classes:")
print(y_raw.value_counts())
X_raw.head()

In [ ]:

y = np.where(y_raw.values.ravel() == 'g', 1, -1)
X = X_raw.values.copy()

print("Classes únicas após recodificação:", np.unique(y))
print(f"  +1 (good): {(y ==  1).sum()} amostras")
print(f"  -1 (bad) : {(y == -1).sum()} amostras")

In [ ]:
df = pd.DataFrame(X, columns=X_raw.columns)

missing = df.isnull().sum().sum()
print(f"Valores ausentes: {missing}")

zero_var = df.columns[df.std() == 0].tolist()
print(f"Atributos constantes (variância = 0): {zero_var}")

if zero_var:
    df.drop(columns=zero_var, inplace=True)
    X = df.values
    print(f"→ Removidos. Novo shape: {X.shape}")
else:
    print("→ Nenhum atributo constante. Shape mantido:", X.shape)

In [ ]:
modelos = {}
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Amostras de treino : {X_train.shape[0]}")
print(f"Amostras de teste  : {X_test.shape[0]}")
print(f"Atributos          : {X_train.shape[1]}")
print(f"\nDistribuição no treino → +1: {(y_train==1).sum()}  | -1: {(y_train==-1).sum()}")
print(f"Distribuição no teste  → +1: {(y_test ==1).sum()}  | -1: {(y_test ==-1).sum()}")

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore', category=ConvergenceWarning)

N_EPOCHS   = 200
SEED       = 42
taxas      = [1e-4, 1e-3, 1e-2, 5e-2, 1e-1]
labels_eta = [r'$\eta = 10^{-4}$', r'$\eta = 10^{-3}$',
              r'$\eta = 10^{-2}$', r'$\eta = 5\times10^{-2}$',
              r'$\eta = 10^{-1}$']

modelos    = {}
historicos = {}

for eta, label in zip(taxas, labels_eta):
    model  = SGDClassifier(loss='squared_error', eta0=eta,
                           learning_rate='constant', max_iter=1,
                           warm_start=True, shuffle=True, random_state=SEED)
    custos = []
    for _ in range(N_EPOCHS):
        model.fit(X_train, y_train)
        u   = model.decision_function(X_train)
        mse = mean_squared_error(y_train, u)
        custos.append(mse if np.isfinite(mse) and mse < 1e6 else np.nan)

    modelos[label]    = model
    historicos[label] = np.array(custos)

valores_finitos = [c for custos in historicos.values()
                   for c in custos if np.isfinite(c)]
ymax = np.percentile(valores_finitos, 95) * 1.2

fig, ax = plt.subplots(figsize=(10, 6))
epochs = range(1, N_EPOCHS + 1)
for label, custos in historicos.items():
    c = np.array(custos, dtype=np.float64)
    c[~np.isfinite(c)] = np.nan
    if np.all(np.isnan(c)):
        ax.plot([], [], linestyle='--', label=label + ' (divergiu)')
    else:
        ax.plot(epochs, c, label=label)

ax.set_ylim(0, ymax)
ax.set_xlabel('Épocas', fontsize=12)
ax.set_ylabel('Custo Médio (MSE)', fontsize=12)
ax.set_title('Convergência do Adaline – Dataset Ionosphere', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim([1, N_EPOCHS])
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\nCusto final (última época):")
for label, custos in historicos.items():
    ultimo = custos[-1]
    print(f"  {label:38s}: {'divergiu' if np.isnan(ultimo) else f'{ultimo:.6f}'}")

In [ ]:

custos_finais = {label: custos[-1] for label, custos in historicos.items()
                 if np.isfinite(custos[-1])}

melhor_label = min(custos_finais, key=custos_finais.get)
melhor_eta   = taxas[labels_eta.index(melhor_label)]

print(f"Melhor η: {melhor_label}  (η = {melhor_eta})")
print(f"Custo final: {custos_finais[melhor_label]:.6f}")

plt.figure(figsize=(8, 5))
plt.plot(epochs, historicos[melhor_label], color='steelblue')
plt.xlabel('Épocas')
plt.ylabel('Custo Médio (MSE)')
plt.title(f'Convergência – Melhor η ({melhor_label})')
plt.xlim([1, N_EPOCHS])
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
adaline_sklearn = SGDClassifier(
    loss='squared_error',
    eta0=melhor_eta,
    learning_rate='constant',
    max_iter=N_EPOCHS,
    shuffle=True,
    random_state=SEED
)
adaline_sklearn.fit(X_train, y_train)
y_pred_sklearn = adaline_sklearn.predict(X_test)

cm_sk = confusion_matrix(y_test, y_pred_sklearn, labels=[-1, 1])
print("Matriz de Confusão (sklearn):")
print(cm_sk)

In [ ]:
print("=== Métricas – Adaline sklearn ===")
print(f"Acurácia  : {accuracy_score(y_test, y_pred_sklearn):.4f}")
print(f"Precisão  : {precision_score(y_test, y_pred_sklearn, pos_label=1):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_sklearn, pos_label=1):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_sklearn, pos_label=1):.4f}")